# 14 Fashion-MNIST MLP：rank 12での更新範囲ablation

Notebook 11で追加候補になったTT-rank 12について、`all_parameters`と`tt_cores_only`を同じepoch 0から比較します。

rank 12は、rank 16より強い圧縮でもaccuracyを回復できるか確認するための境界探索です。Notebook 13では更新方式の優劣がrankによって一貫しなかったため、このNotebookでも両方式を残します。

このNotebookは実装と検証までを扱います。結果の考察と一般化は別の会話で行います。

## 1. 実験で変えるものと固定するもの

比較する更新対象は次の2つです。

$$
\theta_{\mathrm{all}} = \{G_1,G_2,G_3,b_1,W_2,b_2,W_3,b_3\}
$$

$$
\theta_{\mathrm{TT}} = \{G_1,G_2,G_3\}
$$

両方式で、dense checkpoint、tensorization、requested rank、epoch 0のparameter値、seed、mini-batch順、optimizer、学習率、最大epoch数、early-stopping規則を揃えます。

- train 50,000件：fine-tuning
- early-stopping validation 5,000件：checkpoint選択
- rank-selection validation 5,000件：accuracy、agreement、logits RMSEの診断
- test 10,000件：予約のみ。このNotebookでは評価しない

epoch 0は3つ目の学習方式ではなく、両方式に共通する追加学習前の基準です。

In [1]:
# importとプロジェクトルートの解決
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import ast
import copy
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("src/nn_compression が見つかりません。")

from nn_compression.compression import (
    TTLinear,
    factorize_named_tt_linear,
    tt_matrix_ranks,
)
from nn_compression.datasets import (
    get_fashion_mnist_datasets,
    make_fashion_mnist_loaders,
    split_fashion_mnist_dataset,
)
from nn_compression.metrics import agreement, count_parameters, logits_rmse
from nn_compression.models import MNISTMLP
from nn_compression.training import evaluate, train_one_epoch
from nn_compression.utils import (
    find_project_root,
    get_experiment_dirs,
    make_torch_generator,
    set_seed,
)

In [2]:
# Notebook 13と同じ学習条件を固定する。
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
TARGET_LAYER = "fc1"
REQUESTED_RANK = 12
TRAIN_BATCH_SIZE = 64
VALIDATION_BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 50
PATIENCE = 5
MIN_DELTA = 1e-4
SAVE_RESULTS = True

set_seed(SEED)
project_root = find_project_root(Path.cwd())
data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    "30_tt_mps",
    "10_fashion_mnist_mlp",
    "14_rank12_update_scope_ablation",
    create=False,
)
baseline_checkpoint_path = (
    project_root / "models" / "10_svd" / "20_fashion_mnist_mlp"
    / "03_mlp_svd_finetuning_using_src_corrected_rerun"
    / "baseline_same_run.pt"
)
selection_path = (
    project_root / "results" / "30_tt_mps" / "10_fashion_mnist_mlp"
    / "09_pretrained_mlp_fc1_tt_tensorization_comparison"
    / "selected_tensorizations.csv"
)
additional_rank_path = (
    project_root / "results" / "30_tt_mps" / "10_fashion_mnist_mlp"
    / "11_tt_rank_tradeoff_after_finetuning"
    / "additional_rank_requests.csv"
)
comparison13_path = (
    project_root / "results" / "30_tt_mps" / "10_fashion_mnist_mlp"
    / "13_tt_cores_only_ablation" / "update_scope_comparison.csv"
)

for required_path in [
    baseline_checkpoint_path, selection_path, additional_rank_path,
    comparison13_path,
]:
    if not required_path.is_file():
        raise FileNotFoundError(f"必要な入力がありません: {required_path}")

print("device:", DEVICE)
print("results target:", results_dir)

device: cuda
results target: D:\dev\nn-compression-svd-dmrg\results\30_tt_mps\10_fashion_mnist_mlp\14_rank12_update_scope_ablation


## 2. rank 12の由来とtensorizationを確認する

rank 12は結果を見て後付けした値ではなく、Notebook 11が保存した追加候補です。tensorizationはNotebook 09で3つの役割すべてに選ばれた`reversed_sites`を引き継ぎます。

rank 12には`aggressive`などの役割名を新しく付けません。Notebook 08で自動選択された3役割とは別の追加検証だからです。

In [3]:
df_selected09 = pd.read_csv(selection_path)
df_additional_rank = pd.read_csv(additional_rank_path)
df_reference13 = pd.read_csv(comparison13_path)

for column in ["out_modes", "in_modes", "actual_ranks"]:
    if column in df_selected09.columns:
        df_selected09[column] = df_selected09[column].map(
            lambda value: tuple(ast.literal_eval(value))
            if isinstance(value, str) else tuple(value)
        )

assert REQUESTED_RANK in set(df_additional_rank["requested_rank"].astype(int))
assert set(df_selected09["tensorization"]) == {"reversed_sites"}
assert df_selected09["out_modes"].nunique() == 1
assert df_selected09["in_modes"].nunique() == 1

reference_tensorization = df_selected09.iloc[0]
OUT_MODES = tuple(reference_tensorization["out_modes"])
IN_MODES = tuple(reference_tensorization["in_modes"])

rank12_request = df_additional_rank.loc[
    df_additional_rank["requested_rank"].astype(int) == REQUESTED_RANK
].copy()
assert len(rank12_request) == 1

display(rank12_request)
display(df_reference13[[
    "requested_rank", "update_scope", "selected_epoch",
    "rank_validation_accuracy", "prediction_agreement",
    "logits_rmse", "trainable_parameters",
]])
print("tensorization: reversed_sites")
print("out_modes:", OUT_MODES)
print("in_modes:", IN_MODES)

,requested_rank,between_rank_low,between_rank_high,reason
0,12,NaN,16,rank 16のfine-tuning後accuracyがdenseとの差の許容範囲内なので...


,requested_rank,update_scope,selected_epoch,rank_validation_accuracy,prediction_agreement,logits_rmse,trainable_parameters
0,16,all_parameters,5,0.8890,0.9314,3.666437,153386
1,16,tt_cores_only,12,0.8904,0.9186,4.309767,18976
2,32,all_parameters,5,0.8850,0.9298,7.104990,205130
3,32,tt_cores_only,5,0.8926,0.9362,4.571566,70720
4,64,all_parameters,0,0.8872,0.9922,0.179470,406922
5,64,tt_cores_only,0,0.8872,0.9922,0.179470,272512


tensorization: reversed_sites
out_modes: (32, 8, 2)
in_modes: (2, 8, 49)


## 3. データ分割とdense baseline

Notebook 13と同じseed、分割、前処理を再構成します。`test_dataset_reserved`は長さだけを確認し、評価関数へ渡しません。

In [ ]:
set_seed(SEED)
full_train_dataset, test_dataset_reserved = get_fashion_mnist_datasets(
    data_dir, download=False,
)
train_dataset, early_stopping_dataset, rank_validation_dataset = (
    split_fashion_mnist_dataset(
        full_train_dataset, (50_000, 5_000, 5_000), seed=SEED
    )
)
loaders = make_fashion_mnist_loaders(
    train_dataset,
    early_stopping_dataset,
    test_dataset_reserved,
    train_batch_size=TRAIN_BATCH_SIZE,
    validation_batch_size=VALIDATION_BATCH_SIZE,
    test_batch_size=1000,
    rank_validation_dataset=rank_validation_dataset,
    train_generator=make_torch_generator(SEED),
)
early_stopping_loader = loaders["validation_loader"]
rank_validation_loader = loaders["validation_loader_rank"]
criterion = torch.nn.CrossEntropyLoss()

baseline_state = torch.load(
    baseline_checkpoint_path, map_location=DEVICE, weights_only=True,
)
baseline_model = MNISTMLP().to(device=DEVICE, dtype=DTYPE)
baseline_model.load_state_dict(baseline_state)
baseline_model.eval()
baseline_rank_loss, baseline_rank_accuracy = evaluate(
    baseline_model, rank_validation_loader, criterion, DEVICE,
)
baseline_parameters = count_parameters(baseline_model)

assert len(train_dataset) == 50_000
assert len(early_stopping_dataset) == 5_000
assert len(rank_validation_dataset) == 5_000
assert len(test_dataset_reserved) == 10_000
print(
    f"dense rank-validation: loss={baseline_rank_loss:.6f}, "
    f"accuracy={baseline_rank_accuracy:.4f}"
)
print("test set: reserved and not evaluated")

## 4. TODO実装契約

TODOは順番に実装し、1つ終えるたびに入力、出力、禁止事項を照合します。

### TODO 1：rank 12の共通epoch 0と2更新方式を準備する

- 入力：`baseline_state`、`OUT_MODES`、`IN_MODES`、`REQUESTED_RANK`
- 行うこと：dense checkpointからrank 12のTTモデルを1つ作り、deep copyで2方式へ分ける
- 行わないこと：loader使用、forward評価、学習、checkpoint保存
- 出力：`candidate_models[update_scope]`、`optimizers[update_scope]`、`initial_parameter_states[update_scope]`、TT構造と更新対象の記録
- 完了条件：2モデルの全parameter値が完全一致し、違うのは`requires_grad`とoptimizerの対象だけ

### TODO 2：同一条件で2 trialを学習する

- 入力：TODO 1の辞書、3つのdataset／loader、`baseline_model`
- 行うこと：trialごとに同じseedのnew train loaderを作り、epoch 0をinitial bestとして学習する
- 行わないこと：rank-validation指標による選択や停止、test評価、途中の条件変更
- 出力：`epoch0_records`、`history_records`、`selected_records`、`selected_models[update_scope]`
- 完了条件：2 trialが同じearly-stopping規則で選択済みstateへ復元される

### TODO 3：固定parameterとTT構造を検証する

- 入力：選択済みモデル、学習前parameter、初期core shape、initial actual rank
- 行うこと：学習前後の完全一致、最大変化量、core shape、actual rankを検証する
- 行わないこと：再学習、評価指標の再計算
- 出力：`parameter_verification_records`
- 完了条件：`tt_cores_only`の固定parameterがすべて完全一致し、TT構造が不変

### TODO 4〜6：表、図、保存

recordから表を作り、epoch 0と選択済み状態を区別して可視化します。test評価、結果を見た学習条件変更、Notebook 13成果物の上書きは行いません。

## TODO 1：rank 12のepoch 0モデルを準備する

### このセルで確認すること

学習済みdenseモデルの`fc1`を3-core TTLinearへ置き換えます。requested rank 12と、実際に生成されたbond dimensionを混同しないでください。

### 実装方針

1. 新しい`MNISTMLP`へ`baseline_state`を読み込む。
2. `factorize_named_tt_linear`で`fc1`だけをTT化する。
3. `TTLinear`、`tt_matrix_ranks`、core shape、総parameter数を確認する。
4. 共通epoch 0をdeep copyし、2つの独立したモデルを作る。
5. `all_parameters`では全parameter、`tt_cores_only`では`fc1.cores.*`だけを学習可能にする。
6. optimizerには学習可能parameterだけを渡す。
7. 両方式の全parameter値が完全一致することを確認する。

既存の`TTLinear`、`factorize_named_tt_linear`、`tt_matrix_ranks`、`count_parameters`を使い、同じ機能を作り直さないでください。

In [ ]:
# TODO 1: rank 12の共通epoch 0から2つの更新方式を準備する。
# 上の7段階を順に実装し、ここでは学習や評価を行わない。

update_scopes = ("all_parameters", "tt_cores_only")
candidate_models = {}
optimizers = {}
initial_parameter_states = {}
trainable_parameter_records = []
initial_core_shapes = {}
initial_actual_ranks = {}

set_seed(SEED)

# 1. 学習済み重みを読み込むためのdenseモデルを作る。
epoch0_model = MNISTMLP().to(device=DEVICE, dtype=DTYPE)

# 2. 全層を同じ学習済みdense checkpointの値にする。
epoch0_model.load_state_dict(baseline_state)

# 3. fc1だけをTTLinearへ置き換える。
epoch0_model = factorize_named_tt_linear(
    epoch0_model,
    "TARGET_LAYER",
    out_modes=OUT_MODES,
    in_modes=IN_MODES ,
    max_rank=REQUESTED_RANK,
)

# 4. fc1がTTLinearへ置換されたことを確認する。
assert isinstance(epoch0_model.fc1, TTLinear)

# 生成されたTT構造を取得する。
actual_ranks = tt_matrix_ranks(
    list()
)

core_shapes = tuple(
    tuple(core.shape)
    for core in 
)

print("actual_ranks:", actual_ranks)
print("core_shapes:", core_shapes)

## TODO 2：epoch 0を含めて2 trialを学習する

### checkpoint選択

epoch 0のearly-stopping validation lossを最初のbestとします。epoch $t$のlossを$L_t$、現在のbestを$L_{\mathrm{best}}$とすると、更新条件は

$$
L_{\mathrm{best}} - L_t > \delta, \qquad \delta=10^{-4}
$$

です。判定後の`best_updated`と`patience_counter`を各epochの履歴へ保存します。

### 実装方針

- trialごとに`set_seed(SEED)`を呼び、`make_torch_generator(SEED)`を渡したnew train loaderを作る。
- epoch 0と各epochでearly-stopping validationを評価する。
- rank-validation accuracy、agreement、logits RMSEは診断として記録するだけにする。
- best判定とpatience更新の後に履歴を追加する。
- 学習後は必ずbest stateへ戻し、選択済み指標を再評価する。
- epoch 0と選択済み状態を別recordに残し、追加学習の変化を後で計算できるようにする。

`train_one_epoch`、`evaluate`、`agreement`、`logits_rmse`を再利用してください。

In [ ]:
# TODO 2: rank 12 × 2更新方式を同一条件で学習する。
# TODO 1を再実行せず、このセルだけを再実行してはいけない。
epoch0_records = []
history_records = []
selected_records = []
selected_models = {}

# TODO: epoch 0評価、学習、early stopping、best state復元を実装する。
# rank-validation指標とtest setを選択判断に使わないこと。

## TODO 3：固定parameterとTT構造を検証する

### このセルで確認すること

`tt_cores_only`で固定した`fc1.bias`、`fc2`、`fc3`が学習前と完全一致することを確認します。また、学習によってTT coreのshapeやbond dimensionが変わっていないことを確認します。

### 実装方針

1. `selected_models[update_scope]`と`initial_parameter_states[update_scope]`を対応させる。
2. named parameterごとに`torch.equal`と最大絶対変化量を計算する。
3. `requires_grad=False`のparameterは完全一致をassertする。
4. `tt_matrix_ranks`とcore shapeが学習前後で同じことをassertする。
5. parameter名、学習対象か、要素数、完全一致、最大変化量をrecordへ保存する。

In [ ]:
# TODO 3: 学習前後のparameter値とTT構造を検証する。
# このセルでは再学習や評価指標の再計算を行わない。
parameter_verification_records = []

# TODO: 2モデルを走査し、固定parameterとTT構造を検証する。

## TODO 4：比較用DataFrameを作る

次の5表をlong形式で作ります。

- `df_rank12_epoch0`：両方式に共通する追加学習前指標
- `df_rank12_comparison`：2方式の選択済み結果とepoch 0からの変化
- `df_rank12_history`：epoch 0を含む学習履歴
- `df_rank12_trainable_parameters`：更新対象名と要素数
- `df_rank12_parameter_verification`：学習前後のparameter検証

accuracy、agreement、logits RMSEは異なる量です。差分列も指標ごとに分け、混ぜて合成スコアにしません。

In [ ]:
# TODO 4: TODO 1〜3のrecordだけから5つのDataFrameを作る。
# モデルの再評価、条件選択、可視化は行わない。
df_rank12_epoch0 = pd.DataFrame()
df_rank12_comparison = pd.DataFrame()
df_rank12_history = pd.DataFrame()
df_rank12_trainable_parameters = pd.DataFrame()
df_rank12_parameter_verification = pd.DataFrame()

# TODO: 列順、2方式の一意性、epoch 0、値域、固定parameterをassertする。

## TODO 5：可視化する

matplotlibで次を別々に描きます。

1. epoch 0と選択済みrank-validation accuracy
2. epoch 0と選択済みprediction agreement
3. epoch 0と選択済みlogits RMSE
4. 更新方式ごとのtrainable parameter数
5. early-stopping validation lossの学習曲線

accuracyの図だけにdense baselineを表示します。agreementとaccuracyは同じ軸へ重ねません。rankは12だけなので、rankを連続軸にした折れ線グラフにはしません。

In [ ]:
# TODO 5: TODO 4のDataFrameだけを描画元にする。
# Figureは保存セルへ渡せるよう、次の変数名へ格納する。
fig_accuracy = None
fig_agreement = None
fig_logits_rmse = None
fig_trainable_parameters = None
fig_learning_curves = None

# TODO: 5つのFigureを作成する。

## TODO 6：Notebook 14専用成果物を保存する

`SAVE_RESULTS=True`のときだけ、Notebook 14専用ディレクトリへ5 CSV、5 Figure、2 checkpointを保存します。Notebook 13の成果物は上書きも複製もしません。保存セルでは再学習・再評価を行いません。

In [ ]:
# TODO 6: 表、Figure、選択済みstate_dictだけを保存する。
# results_dirとmodels_dirは保存直前に作る。
# test評価や既存成果物の上書きは行わない。

## 5. 実装後のチェックリスト

- [ ] rank 12がNotebook 11の保存済み追加候補から取得されている
- [ ] tensorizationがNotebook 09の`reversed_sites`と一致する
- [ ] 2方式が完全に同じepoch 0 parameterから始まる
- [ ] trialごとに同じseedのnew train loaderを作る
- [ ] epoch 0がbest候補とpatience軌跡へ含まれる
- [ ] checkpoint選択にearly-stopping validation lossだけを使う
- [ ] rank-validation指標を選択・停止・条件変更に使わない
- [ ] `tt_cores_only`の固定parameterが完全一致する
- [ ] actual rankとcore shapeが学習前後で不変である
- [ ] accuracyとagreementを別図にする
- [ ] test setを評価していない
- [ ] Notebook 13の成果物を変更していない

結果を確認した後も、この1 seedだけから更新方式の一般的優劣を断定しません。追加の学習率探索や複数seedは、rank 12の結果から必要性を判断して別Notebookへ分けます。